In [1]:
import pandas as pd

DATA_PATH = "../data/UCI_dataset/"

X_train = pd.read_csv(DATA_PATH + "train/X_train.txt", sep=r'\s+', header=None).values
X_test  = pd.read_csv(DATA_PATH + "test/X_test.txt",  sep=r'\s+', header=None).values

y_train = pd.read_csv(DATA_PATH + "train/y_train.txt", header=None).values.flatten()
y_test  = pd.read_csv(DATA_PATH + "test/y_test.txt",  header=None).values.flatten()


labels_df = pd.read_csv(DATA_PATH + "activity_labels.txt", sep=' ', header=None)
LABELS = labels_df[1].tolist()

print("X_train shape:", X_train.shape)
print("X_test shape:",  X_test.shape)
print("Activités:", LABELS)

X_train shape: (7352, 561)
X_test shape: (2947, 561)
Activités: ['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS', 'SITTING', 'STANDING', 'LAYING']


In [ ]:
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.preprocessing import LabelEncoder
import torch

X_train_tab = X_train
X_test_tab  = X_test

le = LabelEncoder()
y_train_tab = le.fit_transform(y_train)  # [0,1,2,3,4,5]
y_test_tab  = le.transform(y_test)

# Modèle
model_tabnet = TabNetClassifier(
    n_d=64,
    n_a=64,
    n_steps=5,
    gamma=1.5,
    n_independent=2,
    n_shared=2,
    momentum=0.02,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=0.0005),
    mask_type='sparsemax',
    verbose=10
)

model_tabnet.fit(
    X_train_tab, y_train_tab,
    eval_set=[(X_test_tab, y_test_tab)],
    eval_metric=['accuracy'],
    max_epochs=300,
    patience=20,
    batch_size=256,
)

y_pred_tab = model_tabnet.predict(X_test_tab)

from sklearn.metrics import classification_report, accuracy_score
print(f"Accuracy TabNet : {accuracy_score(y_test_tab, y_pred_tab)*100:.2f}%")
print(classification_report(y_test_tab, y_pred_tab))

/home/hiba/HAR_models/.venv/lib/python3.13/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 4.79317 | val_0_accuracy: 0.10112 |  0:00:05s
epoch 10 | loss: 2.19776 | val_0_accuracy: 0.25857 |  0:00:57s
epoch 20 | loss: 1.7467  | val_0_accuracy: 0.35629 |  0:01:54s
epoch 30 | loss: 1.50145 | val_0_accuracy: 0.44045 |  0:03:00s
epoch 40 | loss: 1.26243 | val_0_accuracy: 0.49949 |  0:03:58s
epoch 50 | loss: 1.11957 | val_0_accuracy: 0.54293 |  0:05:02s
epoch 60 | loss: 1.03956 | val_0_accuracy: 0.58568 |  0:05:47s
epoch 70 | loss: 0.93936 | val_0_accuracy: 0.5945  |  0:06:40s
epoch 80 | loss: 0.87783 | val_0_accuracy: 0.62199 |  0:07:39s
epoch 90 | loss: 0.82917 | val_0_accuracy: 0.679   |  0:08:40s
epoch 100| loss: 0.77968 | val_0_accuracy: 0.66882 |  0:09:33s
epoch 110| loss: 0.72336 | val_0_accuracy: 0.69766 |  0:10:27s
epoch 120| loss: 0.72196 | val_0_accuracy: 0.70954 |  0:11:22s
epoch 130| loss: 0.66747 | val_0_accuracy: 0.71564 |  0:12:21s

Early stopping occurred at epoch 138 with best_epoch = 118 and best_val_0_accuracy = 0.72175


/home/hiba/HAR_models/.venv/lib/python3.13/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Accuracy TabNet : 72.18%
              precision    recall  f1-score   support

           0       0.66      0.83      0.73       496
           1       0.62      0.68      0.65       471
           2       0.82      0.48      0.61       420
           3       0.71      0.51      0.59       491
           4       0.66      0.86      0.75       532
           5       0.96      0.90      0.93       537

    accuracy                           0.72      2947
   macro avg       0.74      0.71      0.71      2947
weighted avg       0.74      0.72      0.72      2947

